In [1]:
short_run = True

rows_per_month = 50

max_trees = 40

import csv
import hashlib
import json
import os
import re
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import requests
from lightgbm import LGBMRegressor, early_stopping
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score

sys.dont_write_bytecode = True

start_folder = Path.cwd().resolve()

PROJECT = None

for folder in (start_folder, *start_folder.parents):
    for path in (folder / "app" / "flatfairproj", folder / "flatfairproj", folder):
        if (path / "manage.py").is_file():
            PROJECT = path

            break

    if PROJECT is not None:
        break

if PROJECT is None:
    raise Exception("Run this notebook from inside the FlatFair project folder.")

ROOT = PROJECT.parents[1]

TRAINING = PROJECT / "assets" / "training"

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from storefront.valuation import flagReason, guessBand, rawBand

SALES_FILE = TRAINING / "ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv"

GEO_FOLDER = TRAINING / "geo"

SERVING_MODEL = PROJECT / "assets" / "ml" / "price_model_v5.joblib"

SAVED_METRICS = TRAINING / "price_model_v5_metrics.json"

CHECK_FOLDER = TRAINING / "check_runs" / "price"

MODEL_FOLDER = CHECK_FOLDER if short_run else PROJECT / "assets" / "ml"

REPORT_FOLDER = CHECK_FOLDER if short_run else TRAINING

METRICS_FILE = REPORT_FOLDER / "price_model_v5_metrics.json"

PRICE_BAND_REPORT = REPORT_FOLDER / "price_band_eval.json"

BACKTEST_REPORT = REPORT_FOLDER / "price_backtest_eval.json"

SEED = 42

FLAT_TYPES = ['2 ROOM', '3 ROOM', '4 ROOM', '5 ROOM', 'EXECUTIVE']

CATEGORY_COLUMNS = ['town', 'flat_type', 'flat_model']

PARAMS = {"num_leaves": 127, "min_child_samples": 50, "reg_lambda": 0, "colsample_bytree": 1.0}

NO_GEO_TREES = {"median": 3197, "q10": 1277, "q90": 650}

GEO_COLUMNS = ["lat", "lon", "dist_cbd_km", "dist_mrt_km", "n_mrt_1km"]

SEARCH_URL = "https://www.onemap.gov.sg/api/common/elastic/search"

run_started = time.time()

run_label = "Short training check" if short_run else "Full training run"

print("Mode:", "short run" if short_run else "full run")
print("Sales kept per month:", rows_per_month if short_run else "all")
print("Most trees per model:", max_trees if short_run else "full counts")
print("Model folder:", MODEL_FOLDER)
print("Report folder:", REPORT_FOLDER)

Mode: short run
Sales kept per month: 50
Most trees per model: 40
Model folder: /home/coco/Desktop/c_windows/Users/Koko/Desktop/HELP/25_10_2026/IS2108/flatfair/.flatfair-local/short-names-20261114T144349Z/notebook-after/app/training/check_runs/price
Report folder: /home/coco/Desktop/c_windows/Users/Koko/Desktop/HELP/25_10_2026/IS2108/flatfair/.flatfair-local/short-names-20261114T144349Z/notebook-after/app/training/check_runs/price


In [2]:
session = requests.Session()


def search(query, page=1):
    for attempt in range(5):
        try:
            reply = session.get(
                SEARCH_URL,
                params={"searchVal": query, "returnGeom": "Y", "getAddrDetails": "Y", "pageNum": page},
                timeout=20,
            )

            if reply.status_code == 200:
                return reply.json()

            time.sleep(2**attempt)
        except Exception:
            time.sleep(2**attempt)

    return {"results": [], "totalNumPages": 0}


STREET_ABBREVIATIONS = {
    "AVE": "AVENUE",
    "ST": "STREET",
    "RD": "ROAD",
    "DR": "DRIVE",
    "CRES": "CRESCENT",
    "CTRL": "CENTRAL",
    "NTH": "NORTH",
    "STH": "SOUTH",
    "UPP": "UPPER",
    "LOR": "LORONG",
    "BT": "BUKIT",
    "JLN": "JALAN",
    "TG": "TANJONG",
    "KG": "KAMPONG",
    "PL": "PLACE",
    "CL": "CLOSE",
    "CTR": "CENTRE",
    "HTS": "HEIGHTS",
    "GDNS": "GARDENS",
    "PK": "PARK",
    "TER": "TERRACE",
    "MKT": "MARKET",
    "C'WEALTH": "COMMONWEALTH",
    "NTH.": "NORTH",
    "STH.": "SOUTH",
}


def expand(street):
    words = []

    for word in street.split():
        words.append(STREET_ABBREVIATIONS.get(word, word))

    return " ".join(words)


def geocode(address):
    block, street = address

    whole_road = expand(street)

    for query in (f"{block} {street}", f"{block} {whole_road}"):
        results = search(query).get("results", [])

        for result in results:
            same_block = result.get("BLK_NO", "").upper() == block.upper()

            same_road = result.get("ROAD_NAME", "").upper() == whole_road.upper()

            has_road = whole_road.upper() in result.get("ADDRESS", "").upper()

            if same_block and (same_road or has_road):
                return block, street, result.get("POSTAL", ""), result["LATITUDE"], result["LONGITUDE"], "exact"

        for result in results:
            if result.get("BLK_NO", "").upper() == block.upper():
                return block, street, result.get("POSTAL", ""), result["LATITUDE"], result["LONGITUDE"], "block"

    return block, street, "", "", "", "none"

In [3]:
def get_places(data_path, out_dir):
    os.makedirs(out_dir, exist_ok=True)

    hits = set()

    with open(data_path) as sales_file:
        for row in csv.DictReader(sales_file):
            hits.add((row["block"].strip(), row["street_name"].strip()))

    places = sorted(hits)

    done = {}

    blocks_at = f"{out_dir}/hdb_blocks_geo.csv"

    if os.path.exists(blocks_at):
        with open(blocks_at) as blocks_file:
            for row in csv.DictReader(blocks_file):
                done[(row["block"], row["street_name"])] = row

    todo = [address for address in places if address not in done]

    print("addresses", len(places), "todo", len(todo), flush=True)

    with open(blocks_at, "a", newline="") as blocks_file:
        writer = csv.writer(blocks_file)

        if not done:
            writer.writerow(["block", "street_name", "postal", "lat", "lon", "matched"])

        with ThreadPoolExecutor(6) as pool:
            for i, row in enumerate(pool.map(geocode, todo), 1):
                writer.writerow(row)

                if i % 500 == 0:
                    blocks_file.flush()

                    print("geocoded", i, flush=True)

    stops = {}

    for query in ("MRT STATION", "LRT STATION"):
        first_page = search(query)

        pages = int(first_page.get("totalNumPages", 0) or 0)

        for page in range(1, pages + 1):
            if page == 1:
                data = first_page
            else:
                data = search(query, page)

            for result in data.get("results", []):
                look_for = result.get("SEARCHVAL", "").upper()

                match = re.match(r"^(.*? (MRT|LRT) STATION)", look_for)

                if match and "EXIT" not in look_for:
                    stops.setdefault(match.group(1), (result["LATITUDE"], result["LONGITUDE"]))

    with open(f"{out_dir}/mrt_stations.csv", "w", newline="") as stops_file:
        writer = csv.writer(stops_file)

        writer.writerow(["name", "lat", "lon"])

        for name, (latitude, longitude) in sorted(stops.items()):
            writer.writerow([name, latitude, longitude])

    print("stations", len(stops), flush=True)

    print("DONE", flush=True)

In [4]:
run_location_lookup = False

if run_location_lookup:
    get_places(data_path=SALES_FILE, out_dir=GEO_FOLDER)

In [5]:
def distances(lat, lon, lat2, lon2):
    lat_turn = np.radians(lat)

    lon_turn = np.radians(lon)

    lat2_turn = np.radians(lat2)

    lon2_turn = np.radians(lon2)

    arc = (
        np.sin((lat2_turn - lat_turn) / 2) ** 2
        + np.cos(lat_turn) * np.cos(lat2_turn) * np.sin((lon2_turn - lon_turn) / 2) ** 2
    )

    return 6371 * 2 * np.arcsin(np.sqrt(np.clip(arc, 0, 1)))


def geo_traits(lat, lon, stations):
    lat = np.asarray(lat, dtype=float)

    lon = np.asarray(lon, dtype=float)

    nearest = np.empty(len(lat))

    counts = np.empty(len(lat), dtype=int)

    for start in range(0, len(lat), 2048):
        end = start + 2048

        stop_gaps = distances(lat[start:end, None], lon[start:end, None], stations[:, 0], stations[:, 1])

        nearest[start:end] = stop_gaps.min(axis=1)

        counts[start:end] = (stop_gaps <= 1).sum(axis=1)

    return {
        'lat': lat,
        'lon': lon,
        'dist_cbd_km': distances(lat, lon, 1.2840, 103.8514),
        'dist_mrt_km': nearest,
        'n_mrt_1km': counts,
    }


def metrics(actual, predicted):
    return dict(
        MAE=float(mean_absolute_error(actual, predicted)),
        RMSE=float(mean_squared_error(actual, predicted) ** 0.5),
        MAPE=float(mean_absolute_percentage_error(actual, predicted) * 100),
        R2=float(r2_score(actual, predicted)),
    )


def model(seed, **options):
    if short_run:
        options['n_estimators'] = min(options['n_estimators'], max_trees)

    return LGBMRegressor(n_jobs=4, random_state=seed, verbosity=-1, learning_rate=0.05, **options)


def goal_for(name):
    if name == 'median':
        return {'objective': 'regression_l1'}

    if name == 'q10':
        return {'objective': 'quantile', 'alpha': 0.1}

    return {'objective': 'quantile', 'alpha': 0.9}


def quantile_of(values, weights, quantile):
    order = np.argsort(values)

    in_order = np.asarray(values)[order]

    running = np.cumsum(np.asarray(weights)[order])

    spot = np.searchsorted(running, quantile * np.sum(weights))

    return float(in_order[spot])


def recent_tune(scores, months):
    scores = np.asarray(scores, dtype=float)

    periods = pd.PeriodIndex(months, freq='M').asi8

    weights = 0.8 ** (periods.max() - periods)

    level = min(1, np.ceil((len(scores) + 1) * 0.8) / len(scores))

    pooled = float(np.quantile(scores, level, method='higher'))

    recent = quantile_of(scores, weights, 0.8)

    fix_up = max(0.0, pooled, recent)

    return fix_up, {
        'scheme': 'exponential80_with_pooled80_safeguard',
        'monthly_decay': 0.8,
        'target': 0.8,
        'pooled_offset': pooled,
        'weighted_offset': recent,
        'dollar_offset': fix_up,
        'valid_coverage': float((scores <= fix_up).mean()),
        'weighted_valid_coverage': float(np.average(scores <= fix_up, weights=weights)),
        'calibration_start': str(min(months)),
        'calibration_end': str(max(months)),
    }

In [6]:
def loadData(dataPath):
    data = pd.read_csv(dataPath).drop_duplicates()

    if short_run:
        data = data[data.flat_type.isin(FLAT_TYPES)]

        mixed = data.sample(frac=1, random_state=SEED)

        turn = mixed.groupby(['month', 'town']).cumcount()

        by_turn = mixed.iloc[np.argsort(turn.to_numpy(), kind='stable')]

        picked = by_turn.groupby('month').head(rows_per_month)

        data = data.loc[picked.index.sort_values()]

    return data[data.flat_type.isin(FLAT_TYPES)].copy().reset_index(drop=True)


def makeTraits(data, geoDir, referenceData=None):
    if referenceData is None:
        referenceData = data

    dates = pd.to_datetime(data.month)

    traits = data[CATEGORY_COLUMNS + ["floor_area_sqm"]].copy()

    traits["storey_mid"] = data.storey_range.str.split(" TO ").apply(
        lambda floors: (int(floors[0]) + int(floors[1])) / 2
    )

    traits["sale_year"] = dates.dt.year

    traits["sale_month"] = dates.dt.month

    traits["month_index"] = (dates.dt.year - 2017) * 12 + dates.dt.month - 1

    traits["flat_age"] = dates.dt.year - data.lease_commence_date + (dates.dt.month - 1) / 12

    traits["remaining_lease_months"] = (
        99 * 12 - (dates.dt.year - data.lease_commence_date) * 12 - (dates.dt.month - 1)
    )

    groups = {}

    for column in CATEGORY_COLUMNS:
        groups[column] = sorted(referenceData[column].unique().tolist())

        traits[column] = pd.Categorical(traits[column], categories=groups[column])

    geo = None

    blocks_file = Path(geoDir) / "hdb_blocks_geo.csv"

    stops_file = Path(geoDir) / "mrt_stations.csv"

    if blocks_file.exists() and stops_file.exists():
        blocks = pd.read_csv(blocks_file, dtype={"postal": str, "block": str}).drop_duplicates(["block", "street_name"])

        lat_ok = pd.to_numeric(blocks.lat, errors="coerce").between(1, 2)

        lon_ok = pd.to_numeric(blocks.lon, errors="coerce").between(103, 105)

        blocks = blocks[lat_ok & lon_ok]

        if "matched" in blocks:
            matched = blocks.matched.astype(str).str.lower()

            blocks = blocks[matched.isin(["true", "1", "yes", "exact", "block", "block-level", "other"])]

        places = referenceData[["block", "street_name"]].drop_duplicates()

        share = places.merge(blocks, how="left", on=["block", "street_name"]).lat.notna().mean()

        print("Unique address geo coverage:", share, flush=True)

        stop_table = pd.read_csv(stops_file)

        stops = stop_table[["lat", "lon"]].dropna().to_numpy(float)

        if share >= 0.95 and len(stops):
            referenceJoined = referenceData[["block", "street_name", "town"]].merge(blocks, how="left", on=["block", "street_name"])

            centres = (
                referenceJoined[["block", "street_name", "town", "lat", "lon"]]
                .drop_duplicates(["block", "street_name"])
                .groupby("town")[["lat", "lon"]]
                .mean()
            )

            joined = data[["block", "street_name", "town"]].merge(blocks, how="left", on=["block", "street_name"])

            for column in ["lat", "lon"]:
                joined[column] = joined[column].fillna(joined.town.map(centres[column]))

            if joined[["lat", "lon"]].isna().any().any():
                raise Exception("Missing town centroid")

            place_cols = geo_traits(joined.lat.to_numpy(), joined.lon.to_numpy(), stops)

            for column, values in place_cols.items():
                traits[column] = values

            blocks = blocks.merge(
                referenceData[["block", "street_name", "town"]].drop_duplicates(), on=["block", "street_name"], how="left"
            )

            geo = dict(
                addresses=blocks[["postal", "block", "street_name", "town", "lat", "lon"]].to_dict("records"),
                centroids=centres.to_dict("index"),
                stations=stops,
                coverage=float(share),
                station_names=stop_table["name"].tolist(),
            )

    return traits, groups, geo

In [7]:
def file_hash(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def show_file(label, path):
    if not path.is_file():
        print(label, "not found:", path)

        return None

    found = file_hash(path)

    print(label, path)
    print("  size in bytes:", path.stat().st_size)
    print("  SHA256:", found)
    print("  written after setup ran:", path.stat().st_mtime >= run_started)

    return found


def show_scores(results):
    name = results["selected_variant"] if results["selected_variant"] in results else "v5.2_no_geo"

    test = results[name]["test"]

    band = results[name]["band"]

    print("Chosen model:", results["selected_variant"])
    print("Test MAE in dollars:", round(test["MAE"]))
    print("Test MAPE percent:", round(test["MAPE"], 2))
    print("Test R2:", round(test["R2"], 4))
    print("Test sales inside price range:", round(band["coverage"], 4))
    print("Mean price range width in dollars:", round(band["mean_width"]))
    print("Short run settings:", results.get("short_run", "none, full run"))


def show_model(model_path, metrics_path):
    prices = joblib.load(model_path)

    in_model = json.dumps(prices["metrics"], sort_keys=True)

    in_file = json.dumps(json.loads(metrics_path.read_text()), sort_keys=True)

    print("Model version:", prices.get("version"))
    print("Trees per model:", prices.get("trees"))
    print("Seed:", prices.get("seed"))
    print("Last sales month in model:", prices.get("max_training_month"))
    print("Model metrics match the metrics file:", in_model == in_file)

In [8]:
sales = loadData(SALES_FILE)

print(sales.shape)

print(sales.month.nunique(), "months from", sales.month.min(), "to", sales.month.max())

sales.head()

(5900, 11)
118 months from 2017-01 to 2026-10


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2017-01,ANG MO KIO,3 ROOM,235,ANG MO KIO AVE 3,04 TO 06,67.0,New Generation,1977,59 years 08 months,295000.0
1,2017-01,ANG MO KIO,4 ROOM,463,ANG MO KIO AVE 10,04 TO 06,92.0,New Generation,1980,62 years 01 month,440000.0
2,2017-01,BEDOK,5 ROOM,763,BEDOK RESERVOIR VIEW,16 TO 18,115.0,Premium Apartment,2000,82 years 11 months,610000.0
3,2017-01,BEDOK,5 ROOM,723,BEDOK RESERVOIR RD,04 TO 06,137.0,Model A-Maisonette,1984,66 years 07 months,633000.0
4,2017-01,BISHAN,3 ROOM,105,BISHAN ST 12,04 TO 06,64.0,Simplified,1985,67 years 11 months,395000.0


In [9]:
print("Saved results from an earlier run")

saved_metrics_hash = show_file("Saved metrics file:", SAVED_METRICS)

serving_model_hash = show_file("Serving model:", SERVING_MODEL)

if saved_metrics_hash:
    show_scores(json.loads(SAVED_METRICS.read_text()))

if saved_metrics_hash and serving_model_hash:
    show_model(SERVING_MODEL, SAVED_METRICS)

Saved results from an earlier run
Saved metrics file: /home/coco/Desktop/c_windows/Users/Koko/Desktop/HELP/25_10_2026/IS2108/flatfair/.flatfair-local/short-names-20261114T144349Z/notebook-after/app/training/price_model_v5_metrics.json
  size in bytes: 23146
  SHA256: e60b46778604ffec5f99803245d2efdc20194f4a0675a0f195214fdaa1a5db5c
  written after setup ran: False
Serving model: /home/coco/Desktop/c_windows/Users/Koko/Desktop/HELP/25_10_2026/IS2108/flatfair/.flatfair-local/short-names-20261114T144349Z/notebook-after/app/flatfairproj/ml/price_model_v5.joblib
  size in bytes: 29502418
  SHA256: 49db3c767892c5c0418bdd28eb74d67e3dab879262b11dcbe960988e03aa6cf7
  written after setup ran: False
Chosen model: v5.3
Test MAE in dollars: 29104
Test MAPE percent: 4.31
Test R2: 0.96
Test sales inside price range: 0.7112
Mean price range width in dollars: 77409
Short run settings: none, full run


Model version: v5.3
Trees per model: {'median': 3981, 'q10': 1435, 'q90': 1184}
Seed: 42
Last sales month in model: 2026-10
Model metrics match the metrics file: True


In [10]:
# Teach the home price models
def train_cost(data_path, geo_dir, out_dir, metrics_path, seed=42):
    data = loadData(data_path)

    traits, groups, geo = makeTraits(data, geo_dir)

    train = data.month <= '2025-04'

    valid = data.month.between('2025-05', '2025-10')

    test = data.month.between('2025-11', '2026-10')

    log_costs = np.log(data.resale_price.to_numpy())

    variant = 'v5.3' if geo else 'v5.2'

    plain_traits = traits.drop(columns=GEO_COLUMNS, errors='ignore')

    old_models = {}

    models = {}

    trees = {}

    for name, count in NO_GEO_TREES.items():
        goal_picks = goal_for(name)

        old_model = model(seed, n_estimators=count, **goal_picks, **PARAMS)

        old_models[name] = old_model.fit(plain_traits[train], log_costs[train])

        fitted = model(seed, n_estimators=4000, **goal_picks, **PARAMS)

        fitted.fit(
            traits[train],
            log_costs[train],
            eval_set=[(traits[valid], log_costs[valid])],
            eval_metric='l1' if name == 'median' else 'quantile',
            callbacks=[early_stopping(100, verbose=False)],
        )

        models[name] = fitted

        trees[name] = fitted.best_iteration_

        print(variant, name, 'VALID-selected trees', trees[name], flush=True)

    def predict(fitted_models, mask, table=None):
        if table is None:
            table = traits

        guessed = {}

        for ai_name, trained in fitted_models.items():
            guessed[ai_name] = np.exp(trained.predict(table[mask]))

        return guessed

    def cqr_scores(predicted, target):
        lower = np.minimum(predicted['q10'], predicted['q90'])

        upper = np.maximum(predicted['q10'], predicted['q90'])

        return np.maximum(lower - target, target - upper)

    def bounds(predicted, correction):
        point = predicted['median']

        lower = np.maximum(1, np.minimum(np.minimum(predicted['q10'], predicted['q90']) - correction, point - 1))

        upper = np.maximum(np.maximum(predicted['q10'], predicted['q90']) + correction, point + 1)

        return lower, upper

    def band_metrics(target, low, high):
        return {
            'coverage': float(((target >= low) & (target <= high)).mean()),
            'mean_width': float((high - low).mean()),
        }

    valid_costs = data.resale_price[valid].to_numpy()

    valid_guess = predict(models, valid)

    scores = cqr_scores(valid_guess, valid_costs)

    fix_up, tuning = recent_tune(scores, data.month[valid])

    old_scores = cqr_scores(predict(old_models, valid, plain_traits), valid_costs)

    old_fix, old_tuning = recent_tune(old_scores, data.month[valid])

    print('Locked VALID calibration', tuning, flush=True)

    legacy = json.loads((TRAINING / 'legacy_mappings.json').read_text())

    old_traits = traits[['floor_area_sqm', 'sale_year', 'sale_month', 'storey_mid']].copy()

    lease = data.remaining_lease.str.extract(r'(\d+) years?(?:\s+(\d+) months?)?').fillna('0').astype(float)

    old_traits['remaining_lease_years'] = lease[0] + lease[1] / 12

    for region in ['East', 'North', 'North-East', 'West']:
        old_traits['region_' + region] = data.town.isin(legacy['region_mapping'][region]).astype(int)

    for group in ['Large', 'Premium']:
        old_traits['flat_model_bin_' + group] = data.flat_model.isin(legacy['flat_model_mapping'][group]).astype(int)

    old_traits['flat_type_encoded'] = data.flat_type.map(legacy['flat_type_order'])

    old_traits['storey_bin_encoded'] = np.select([traits.storey_mid <= 6, traits.storey_mid <= 15], [0, 1], default=2)

    for column in ['town', 'flat_model']:
        old_traits[column + '_encoded'] = pd.Categorical(data[column], categories=groups[column]).codes

    base = model(seed, n_estimators=1000, num_leaves=255).fit(old_traits[train], data.resale_price[train])

    test_guess = predict(models, test)

    old_guesses = predict(old_models, test, plain_traits)

    guesses = {
        'old': base.predict(old_traits[test]),
        'v5.2_no_geo': old_guesses['median'],
        variant: test_guess['median'],
    }

    bands = {
        'old': (guesses['old'] * 0.928, guesses['old'] * 1.074),
        'v5.2_no_geo': bounds(old_guesses, old_fix),
        variant: bounds(test_guess, fix_up),
    }

    results = {}

    train_guess = {
        'old': base.predict(old_traits[train]),
        'v5.2_no_geo': predict(old_models, train, plain_traits)['median'],
        variant: predict(models, train)['median'],
    }

    test_costs = data.resale_price[test].to_numpy()

    for name, point in guesses.items():
        test_marks = metrics(test_costs, point)

        train_marks = metrics(data.resale_price[train], train_guess[name])

        low, high = bands[name]

        results[name] = {
            'train': train_marks,
            'test': test_marks,
            'band': band_metrics(test_costs, low, high),
            'MAE_gap': test_marks['MAE'] - train_marks['MAE'],
            'MAPE_gap_pp': test_marks['MAPE'] - train_marks['MAPE'],
        }

    results['per_town'] = {}

    for town in sorted(data.town.unique()):
        in_town = data.town[test].to_numpy() == town

        town_done = {}

        for name, point in guesses.items():
            low, high = bands[name]

            town_done[name] = {
                **metrics(test_costs[in_town], point[in_town]),
                **band_metrics(test_costs[in_town], low[in_town], high[in_town]),
            }

        results['per_town'][town] = town_done

    ship_geo = bool(geo) and results[variant]['test']['MAPE'] < results['v5.2_no_geo']['test']['MAPE']

    results['selected_variant'] = variant if ship_geo or not geo else 'v5.2'

    results['geo_coverage'] = geo['coverage'] if geo else 0.0

    gains = models['median'].booster_.feature_importance(importance_type='gain')

    gain_pairs = []

    for column, gain in zip(traits.columns, gains):
        gain_pairs.append((column, float(gain)))

    gain_pairs.sort(key=lambda pair: pair[1], reverse=True)

    results['feature_gain'] = dict(gain_pairs)

    results['selection_rule'] = (
        'Trees are picked by early stopping on VALID and the range is calibrated on VALID. '
        'TEST is only used once, to check that the location features help before saving the model.'
    )

    results['calibration'] = tuning

    valid_low, valid_high = bounds(valid_guess, fix_up)

    results['band'] = {
        **results[variant]['band'],
        'valid_coverage': band_metrics(valid_costs, valid_low, valid_high)['coverage'],
    }

    print('Frozen TEST comparison', json.dumps({name: results[name] for name in guesses}), flush=True)

    roll_done = []

    for end in ['2022-04', '2023-04', '2024-04']:
        origin = pd.Period(end, 'M')

        history = data.month <= end

        holdout = data.month.between(str(origin + 1), str(origin + 6))

        roll_model = model(seed, n_estimators=trees['median'], objective='regression_l1', **PARAMS)

        roll_model.fit(traits[history], log_costs[history])

        spare_guess = np.exp(roll_model.predict(traits[holdout]))

        roll_done.append({'train_end': end, **metrics(data.resale_price[holdout], spare_guess)})

    results['rolling_cv'] = roll_done

    if short_run:
        results['short_run'] = dict(rows_per_month=rows_per_month, max_trees=max_trees)

    for name in models:
        last_model = model(seed, n_estimators=trees[name], **goal_for(name), **PARAMS)

        models[name] = last_model.fit(traits, log_costs)

    out_folder = Path(out_dir)

    out_folder.mkdir(parents=True, exist_ok=True)

    bundle = dict(
        version=variant,
        selected_variant=variant,
        models=models,
        categories=groups,
        features=list(traits.columns),
        calibration_offsets={'lower': fix_up, 'upper': fix_up},
        calibration_scale='dollars',
        max_training_month=data.month.max(),
        metrics=results,
        params=PARAMS,
        trees=trees,
        geo=geo,
        seed=seed,
        legacy_mappings=legacy,
    )

    if short_run or not geo or ship_geo:
        joblib.dump(bundle, out_folder / 'price_model_v5.joblib', compress=3)

    Path(metrics_path).write_text(json.dumps(results, indent=2))

    print(json.dumps({'params': PARAMS, 'trees': trees, 'metrics': results}, indent=2), flush=True)

In [11]:
print(run_label)

if short_run:
    CHECK_FOLDER.mkdir(parents=True, exist_ok=True)

train_cost(
    data_path=SALES_FILE,
    geo_dir=GEO_FOLDER,
    out_dir=MODEL_FOLDER,
    metrics_path=METRICS_FILE,
    seed=SEED,
)

Short training check


Unique address geo coverage: 0.9796391752577319


v5.3 median VALID-selected trees 40


v5.3 q10 VALID-selected trees 40


/home/coco/Desktop/c_windows/Users/Koko/Desktop/HELP/25_10_2026/IS2108/flatfair/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/home/coco/Desktop/c_windows/Users/Koko/Desktop/HELP/25_10_2026/IS2108/flatfair/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


v5.3 q90 VALID-selected trees 40


Locked VALID calibration {'scheme': 'exponential80_with_pooled80_safeguard', 'monthly_decay': 0.8, 'target': 0.8, 'pooled_offset': -2309.914486159687, 'weighted_offset': -4671.083010278875, 'dollar_offset': 0.0, 'valid_coverage': 0.8133333333333334, 'weighted_valid_coverage': 0.8189279208951341, 'calibration_start': '2025-05', 'calibration_end': '2025-10'}


/home/coco/Desktop/c_windows/Users/Koko/Desktop/HELP/25_10_2026/IS2108/flatfair/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Frozen TEST comparison {"old": {"train": {"MAE": 45451.25023067838, "RMSE": 62480.538510729035, "MAPE": 9.09554436858767, "R2": 0.9027047189345672}, "test": {"MAE": 86134.7647338726, "RMSE": 130461.74090554101, "MAPE": 11.093566074277769, "R2": 0.7118676466900269}, "band": {"coverage": 0.43833333333333335, "mean_width": 91915.02230014012}, "MAE_gap": 40683.514503194216, "MAPE_gap_pp": 1.9980217056900997}, "v5.2_no_geo": {"train": {"MAE": 49124.99467629067, "RMSE": 76010.6966406646, "MAPE": 8.999912965178954, "R2": 0.8560035866947617}, "test": {"MAE": 97156.66540068673, "RMSE": 150055.43457996412, "MAPE": 11.966693689701232, "R2": 0.6188208430675661}, "band": {"coverage": 0.7966666666666666, "mean_width": 329808.57561928575}, "MAE_gap": 48031.67072439606, "MAPE_gap_pp": 2.9667807245222786}, "v5.3": {"train": {"MAE": 45971.11482498269, "RMSE": 73603.76545418763, "MAPE": 8.382962630804938, "R2": 0.8649786908778863}, "test": {"MAE": 99467.4774670466, "RMSE": 153164.43737800667, "MAPE": 12.

{
  "params": {
    "num_leaves": 127,
    "min_child_samples": 50,
    "reg_lambda": 0,
    "colsample_bytree": 1.0
  },
  "trees": {
    "median": 40,
    "q10": 40,
    "q90": 40
  },
  "metrics": {
    "old": {
      "train": {
        "MAE": 45451.25023067838,
        "RMSE": 62480.538510729035,
        "MAPE": 9.09554436858767,
        "R2": 0.9027047189345672
      },
      "test": {
        "MAE": 86134.7647338726,
        "RMSE": 130461.74090554101,
        "MAPE": 11.093566074277769,
        "R2": 0.7118676466900269
      },
      "band": {
        "coverage": 0.43833333333333335,
        "mean_width": 91915.02230014012
      },
      "MAE_gap": 40683.514503194216,
      "MAPE_gap_pp": 1.9980217056900997
    },
    "v5.2_no_geo": {
      "train": {
        "MAE": 49124.99467629067,
        "RMSE": 76010.6966406646,
        "MAPE": 8.999912965178954,
        "R2": 0.8560035866947617
      },
      "test": {
        "MAE": 97156.66540068673,
        "RMSE": 150055.43457996412,


In [12]:
def band_stats(target, rawBounds, servedBounds, flags):
    low, worth, high = servedBounds.T

    errors = np.abs(worth - target) / target * 100

    width = high - low

    return {
        "count": len(target),
        "servedCoverage": float(((target >= low) & (target <= high)).mean()),
        "rawCoverage": float(((target >= rawBounds[:, 0]) & (target <= rawBounds[:, 2])).mean()),
        "meanWidthDollars": float(width.mean()),
        "meanWidthPercent": float((width / worth * 100).mean()),
        "mapePercent": float(errors.mean()),
        "medianAbsolutePercentError": float(np.median(errors)),
        "flaggedShare": float(np.mean([reason is not None for reason in flags])),
        "aboveShare": float((flags == "above").mean()),
        "typoShare": float((flags == "typo").mean()),
    }


# Check the home price range
def try_band(data_path, geo_dir, out_path):
    data = loadData(data_path)

    data = data[data.month <= "2026-10"].reset_index(drop=True)

    history = data.month <= "2025-10"

    train = data.month <= "2025-04"

    valid = data.month.between("2025-05", "2025-10")

    checkup = data.month.between("2025-11", "2026-10")

    if not (train.any() and valid.any() and checkup.any()):
        raise Exception("Training, calibration and evaluation periods must contain sales")

    traits, groups, geo = makeTraits(data, geo_dir, referenceData=data[history])

    if geo is None:
        raise Exception("The fixed v5.3 configuration requires local geographic data")

    trees = {"median": 3981, "q10": 1435, "q90": 1184}

    if short_run:
        for name in trees:
            trees[name] = min(trees[name], max_trees)

    target = data.resale_price.to_numpy(float)

    logGoal = np.log(target)

    validGuess = {}

    for name, count in trees.items():
        fitted = model(42, n_estimators=count, **goal_for(name), **PARAMS)

        fitted.fit(traits[train], logGoal[train])

        validGuess[name] = np.exp(fitted.predict(traits[valid]))

        print("Calibration fit", name, count, flush=True)

        del fitted

    scores = np.maximum(
        np.minimum(validGuess["q10"], validGuess["q90"]) - target[valid],
        target[valid] - np.maximum(validGuess["q10"], validGuess["q90"]),
    )

    fix_up, tuning = recent_tune(scores, data.month[valid])

    offsets = {"lower": fix_up, "upper": fix_up}

    checkTraits = traits[checkup].copy()

    checkTraits["sale_year"] = 2025

    checkTraits["sale_month"] = 10

    checkTraits["month_index"] = (2025 - 2017) * 12 + 9

    guesses = {}

    for name, count in trees.items():
        fitted = model(42, n_estimators=count, **goal_for(name), **PARAMS)

        fitted.fit(traits[history], logGoal[history])

        guesses[name] = fitted.predict(checkTraits)

        print("Historical refit", name, count, flush=True)

        del fitted

    rows = []

    for median, q10, q90 in zip(guesses["median"], guesses["q10"], guesses["q90"]):
        rows.append({"median": median, "q10": q10, "q90": q90})

    rawBounds = np.asarray([rawBand(row, offsets) for row in rows])

    servedBounds = np.asarray([guessBand(row, offsets) for row in rows])

    checkGoal = target[checkup]

    flags = np.asarray([
        flagReason(price, *band) for price, band in zip(checkGoal, servedBounds)
    ], dtype=object)

    report = {
        "schemaVersion": 1,
        "evaluationPeriod": {"start": "2025-11", "end": "2026-10"},
        "trainingPeriod": {"start": str(data.month[history].min()), "end": "2025-10", "count": int(history.sum())},
        "modelConfig": {
            "variant": "v5.3",
            "seed": 42,
            "learningRate": 0.05,
            "nJobs": 4,
            "params": PARAMS,
            "trees": trees,
            "objectives": {name: goal_for(name) for name in trees},
            "features": list(traits.columns),
            "categories": groups,
            "geoCoverage": geo["coverage"],
            "choices": "Fixed shipped v5.3 features and VALID-selected tree counts; no evaluation-period selection or early stopping in this script.",
            "calibrationFitEnd": "2025-04",
            "refitEnd": "2025-10",
            "trendCap": "2025-10; flat age and remaining lease use each sale month, as in serving",
        },
        "calibration": tuning,
        "definitions": {
            "rawCoverage": "Ordered q10/q90 interval with historical dollar calibration, before serving rounding and minimum width.",
            "meanWidthPercent": "Mean of (served high - served low) / rounded estimate * 100.",
            "flaggedShare": "Real sale price treated as asking price; above high or strictly below 70% of low. Proxy for false holds, not observed moderation errors.",
        },
        "limitations": [
            "This period previously informed the shipped location-feature choice. This fixed historical replay avoids new selection but cannot create a fresh independent holdout.",
            "Uses the local static block and station files, not historical geographic snapshots.",
            "Evaluates a temporary historical refit, not the served artifact refit through October 2026.",
        ],
        "overall": band_stats(checkGoal, rawBounds, servedBounds, flags),
    }

    for column, key in (("town", "byTown"), ("flat_type", "byFlatType")):
        values = data.loc[checkup, column].to_numpy()

        report[key] = {}

        for name in sorted(set(values)):
            in_group = values == name

            report[key][name] = band_stats(
                checkGoal[in_group], rawBounds[in_group], servedBounds[in_group], flags[in_group]
            )

    if short_run:
        report["shortRun"] = dict(rowsPerMonth=rows_per_month, maxTrees=max_trees)

    Path(out_path).write_text(json.dumps(report, indent=2, allow_nan=False) + "\n", encoding="utf-8")

    print(json.dumps(report["overall"], indent=2), flush=True)

In [13]:
print(run_label)

if short_run:
    CHECK_FOLDER.mkdir(parents=True, exist_ok=True)

try_band(
    data_path=SALES_FILE,
    geo_dir=GEO_FOLDER,
    out_path=PRICE_BAND_REPORT,
)

Short training check


Unique address geo coverage: 0.9789939192924267


Calibration fit median 40


Calibration fit q10 40


Calibration fit q90 40


Historical refit median 40


Historical refit q10 40


Historical refit q90 40


{
  "count": 600,
  "servedCoverage": 0.8016666666666666,
  "rawCoverage": 0.8,
  "meanWidthDollars": 320003.98333333334,
  "meanWidthPercent": 51.25716454830971,
  "mapePercent": 10.861129559206637,
  "medianAbsolutePercentError": 8.161444072948328,
  "flaggedShare": 0.17833333333333334,
  "aboveShare": 0.17833333333333334,
  "typoShare": 0.0
}


In [14]:
def period(data):
    return {"start": str(data.month.min()), "end": str(data.month.max()), "count": len(data)}


def cap_trend(features, end):
    features = features.copy()

    cutoff = pd.Period(end, freq="M")

    features["sale_year"] = cutoff.year

    features["sale_month"] = cutoff.month

    features["month_index"] = (cutoff.year - 2017) * 12 + cutoff.month - 1

    return features


def past_stats(rows):
    return {
        "count": len(rows),
        "servedCoverage": float(rows.servedInside.mean()),
        "rawCoverage": float(rows.rawInside.mean()),
        "medianWidthPercent": float(rows.widthPercent.median()),
        "meanWidthPercent": float(rows.widthPercent.mean()),
        "meanWidthDollars": float(rows.widthDollars.mean()),
        "mapePercent": float(rows.absolutePercentError.mean()),
        "falseHoldRate": float(rows.held.mean()),
        "aboveShare": float(rows.above.mean()),
        "typoShare": float(rows.typo.mean()),
    }


def group_stats(rows):
    groups = {}

    weakest = []

    for column, key in (("town", "byTown"), ("flat_type", "byFlatType")):
        groups[key] = {}

        for name, subset in rows.groupby(column, observed=True, sort=True):
            result = past_stats(subset)

            groups[key][name] = result

            if result["count"] >= 30:
                weakest.append({"dimension": column, "group": name, **result})

    groups["weakestSubgroups"] = sorted(
        weakest, key=lambda group: (group["servedCoverage"], group["dimension"], group["group"])
    )[:5]

    return groups


def fit_inner(name, variant, train_x, valid_x, train_y, valid_y, seed):
    options = {}

    count = NO_GEO_TREES[name] if variant == "v5.2_no_geo" else 4000

    if short_run:
        count = min(count, max_trees)

    if variant == "v5.3":
        options = {
            "eval_set": [(valid_x, valid_y)],
            "eval_metric": "l1" if name == "median" else "quantile",
            "callbacks": [early_stopping(100, verbose=False)],
        }

    fitted = model(seed, n_estimators=count, **goal_for(name), **PARAMS)

    fitted.fit(train_x, train_y, **options)

    trees = count if variant == "v5.2_no_geo" else int(fitted.best_iteration_)

    guesses = fitted.predict(valid_x)

    print("Inner fit", variant, name, trees, flush=True)

    return guesses, trees

In [15]:
# Test one past price group
def try_fold(data, start, end, geo_dir, seed):
    started = time.perf_counter()

    outer_train = data[data.month < start].reset_index(drop=True)

    outer_test = data[data.month.between(start, end)].reset_index(drop=True)

    inner_start = str(pd.Period(start, freq="M") - 6)

    inner_train = outer_train[outer_train.month < inner_start].reset_index(drop=True)

    inner_valid = outer_train[outer_train.month >= inner_start].reset_index(drop=True)

    if inner_train.empty or inner_valid.month.nunique() != 6 or outer_test.month.nunique() != 6:
        raise Exception("Each fold needs training history and six complete inner and outer months")

    print("Outer test", start, end, "inner validation", inner_start, str(outer_train.month.max()), flush=True)

    inner_data = pd.concat([inner_train, inner_valid], ignore_index=True)

    traits, inner_kinds, inner_geo = makeTraits(inner_data, geo_dir, referenceData=inner_train)

    if inner_geo is None:
        raise Exception("Both original feature candidates require the local geographic data")

    train_x = traits.iloc[:len(inner_train)]

    valid_x = cap_trend(traits.iloc[len(inner_train):], str(inner_train.month.max()))

    train_y = np.log(inner_train.resale_price.to_numpy(float))

    valid_goal = inner_valid.resale_price.to_numpy(float)

    valid_y = np.log(valid_goal)

    pool = {}

    for variant in ("v5.2_no_geo", "v5.3"):
        if variant == "v5.3":
            columns = list(traits.columns)
        else:
            columns = [column for column in traits.columns if column not in GEO_COLUMNS]

        guess, count = fit_inner("median", variant, train_x[columns], valid_x[columns], train_y, valid_y, seed)

        guess_costs = np.exp(guess)

        pool[variant] = {
            "medianMapePercent": float(np.mean(np.abs(guess_costs - valid_goal) / valid_goal) * 100),
            "medianTrees": count,
            "features": columns,
        }

    if pool["v5.3"]["medianMapePercent"] < pool["v5.2_no_geo"]["medianMapePercent"]:
        chosen = "v5.3"
    else:
        chosen = "v5.2_no_geo"

    columns = pool[chosen]["features"]

    trees = {"median": pool[chosen]["medianTrees"]}

    levels = {}

    for name in ("q10", "q90"):
        guess, count = fit_inner(name, chosen, train_x[columns], valid_x[columns], train_y, valid_y, seed)

        levels[name] = guess

        trees[name] = count

    scores = np.maximum(
        np.exp(np.minimum(levels["q10"], levels["q90"])) - valid_goal,
        valid_goal - np.exp(np.maximum(levels["q10"], levels["q90"])),
    )

    fix_up, tuning = recent_tune(scores, inner_valid.month)

    offsets = {"lower": fix_up, "upper": fix_up}

    del traits, train_x, valid_x, inner_data, levels

    refit_data = pd.concat([outer_train, outer_test], ignore_index=True)

    refit_x, groups, geo = makeTraits(refit_data, geo_dir, referenceData=outer_train)

    if geo is None:
        raise Exception("Historical refit lost geographic coverage")

    full_train_x = refit_x.iloc[:len(outer_train)][columns]

    test_x = cap_trend(refit_x.iloc[len(outer_train):][columns], str(outer_train.month.max()))

    full_train_y = np.log(outer_train.resale_price.to_numpy(float))

    guesses = {}

    for name, count in trees.items():
        fitted = model(seed, n_estimators=count, **goal_for(name), **PARAMS)

        fitted.fit(full_train_x, full_train_y)

        guesses[name] = fitted.predict(test_x)

        del fitted

        print("Outer refit", start, chosen, name, count, flush=True)

    guess_rows = []

    for median, q10, q90 in zip(guesses["median"], guesses["q10"], guesses["q90"]):
        guess_rows.append({"median": median, "q10": q10, "q90": q90})

    raw_bounds = np.asarray([rawBand(row, offsets) for row in guess_rows])

    served_bounds = np.asarray([guessBand(row, offsets) for row in guess_rows])

    target = outer_test.resale_price.to_numpy(float)

    low, worth, high = served_bounds.T

    flags = np.asarray([flagReason(price, *band) for price, band in zip(target, served_bounds)], dtype=object)

    rows = outer_test[["month", "town", "flat_type"]].copy()

    rows["servedInside"] = (target >= low) & (target <= high)

    rows["rawInside"] = (target >= raw_bounds[:, 0]) & (target <= raw_bounds[:, 2])

    rows["widthDollars"] = high - low

    rows["widthPercent"] = (high - low) / worth * 100

    rows["absolutePercentError"] = np.abs(worth - target) / target * 100

    rows["held"] = [reason is not None for reason in flags]

    rows["above"] = flags == "above"

    rows["typo"] = flags == "typo"

    report = {
        "testPeriod": period(outer_test),
        "trainingPeriod": period(outer_train),
        "innerTrainingPeriod": period(inner_train),
        "innerValidationPeriod": period(inner_valid),
        "selectedVariant": chosen,
        "innerCandidates": pool,
        "trees": trees,
        "calibration": tuning,
        "calibrationOffsets": offsets,
        "trendCap": {"inner": str(inner_train.month.max()), "outer": str(outer_train.month.max())},
        "innerCategories": inner_kinds,
        "refitCategories": groups,
        "geoCoverage": {"inner": inner_geo["coverage"], "refit": geo["coverage"]},
        "overall": past_stats(rows),
        **group_stats(rows),
        "runtimeSeconds": time.perf_counter() - started,
    }

    print("Completed", start, json.dumps(report["overall"]), "seconds", round(report["runtimeSeconds"], 2), flush=True)

    return report, rows

In [16]:
# Check past home price guesses
def try_prices(data_path, geo_dir, out_path, seed=42):
    started = time.perf_counter()

    data = loadData(data_path)

    latest = pd.Period(data.month.max(), freq="M")

    last_end = min(latest, pd.Period("2026-10", freq="M"))

    first_start = pd.Period("2024-11", freq="M")

    if last_end < pd.Period("2026-04", freq="M"):
        first_start = last_end - 17

    folds = []

    start = first_start

    while start + 5 <= last_end:
        folds.append((str(start), str(start + 5)))

        start += 6

    if len(folds) < 3:
        raise Exception("At least three complete outer folds are required")

    old_hash = file_hash(SERVING_MODEL)

    report = {
        "schemaVersion": 1,
        "dataPeriod": period(data),
        "modelConfig": {
            "seed": seed,
            "nJobs": 4,
            "learningRate": 0.05,
            "params": PARAMS,
            "candidateVariants": ["v5.2_no_geo", "v5.3"],
            "noGeoTrees": NO_GEO_TREES,
            "geoTreeSelection": "At most 4000 trees with 100-round early stopping on each inner validation period, as in the trainer.",
            "featureSelection": "Choose geography only when inner median-price MAPE is strictly lower; the old notebook model is a diagnostic baseline, not a shipping candidate.",
            "calibration": "Trainer recency_calibration on chosen candidate inner-validation quantile residuals only; retain these dollar offsets after refitting on full outer training history.",
            "preprocessing": "Inner categories, geo-coverage gate and town centroids use inner training only; refit preprocessing uses full outer training only. Trend is capped at each fit cutoff, while age and lease use sale month.",
            "search": "Original candidate settings only; no hyperparameter grid and no deployed tree counts reused for the geographic candidate.",
        },
        "definitions": {
            "servedCoverage": "Fraction of actual prices inside the inclusive serving predictionBand after dollar calibration, rounding and minimum widening.",
            "rawCoverage": "Fraction inside serving rawBand: ordered quantiles plus inner-only dollar calibration, before rounding and minimum widening.",
            "medianWidthPercent": "Median of each full served band width divided by its rounded estimate, times 100.",
            "meanWidthPercent": "Mean of each full served band width divided by its rounded estimate, times 100.",
            "mapePercent": "Mean absolute percentage error of the rounded serving median estimate.",
            "falseHoldRate": "Fraction of actual sale prices flagged by serving flagReason when treated as asking prices: above high or strictly below 70% of low; a proxy, not observed moderation errors.",
            "pooled": "Concatenate disjoint outer-test sales once; compute row-weighted metrics, including the median width from all rows, rather than averaging fold summaries.",
            "weakestSubgroups": "Five lowest served-coverage town or flat-type marginal groups with at least 30 outer sales, pooled or within each fold; full subgroup metrics are also retained.",
        },
        "limitations": [
            "Historical nested evaluation of the existing candidate family, not a fresh prospective holdout or a guarantee of future coverage.",
            "The original no-geography candidate retains its trainer's fixed tree counts; the existing candidate family and structural hyperparameters were developed previously, not selected anew here.",
            "Inner validation is reused for feature/tree selection and calibration; outer tests provide evaluation only, and earlier outer sales become history in later rolling folds.",
            "Local static block and MRT files are used, not historical geography snapshots.",
            "The deployed model was refit on all data through October 2026; this script evaluates temporary historical refits and never replaces the artifact.",
        ],
        "folds": [],
    }

    all_rows = []

    for start, end in folds:
        result, rows = try_fold(data, start, end, geo_dir, seed)

        report["folds"].append(result)

        all_rows.append(rows)

    pooled = pd.concat(all_rows, ignore_index=True)

    report["pooled"] = {"testPeriod": period(pooled), "overall": past_stats(pooled), **group_stats(pooled)}

    end_hash = file_hash(SERVING_MODEL)

    if end_hash != old_hash:
        raise Exception("Deployed price artifact changed during evaluation")

    report["deployedArtifact"] = {"sha256Before": old_hash, "sha256After": end_hash, "unchanged": True}

    report["runtimeSeconds"] = time.perf_counter() - started

    if short_run:
        report["shortRun"] = dict(rowsPerMonth=rows_per_month, maxTrees=max_trees)

    Path(out_path).write_text(json.dumps(report, indent=2, allow_nan=False) + "\n", encoding="utf-8")

    summary = {
        "pooled": report["pooled"]["overall"],
        "weakestSubgroups": report["pooled"]["weakestSubgroups"],
        "runtimeSeconds": report["runtimeSeconds"],
    }

    print(json.dumps(summary, indent=2), flush=True)

In [17]:
print(run_label)

if short_run:
    CHECK_FOLDER.mkdir(parents=True, exist_ok=True)

try_prices(
    data_path=SALES_FILE,
    geo_dir=GEO_FOLDER,
    out_path=BACKTEST_REPORT,
    seed=SEED,
)

Short training check


Outer test 2024-11 2025-04 inner validation 2024-05 2024-10


Unique address geo coverage: 0.978254018279231


Inner fit v5.2_no_geo median 40


Inner fit v5.3 median 40


/tmp/ipykernel_960196/4261708868.py:49: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  traits[column] = pd.Categorical(traits[column], categories=groups[column])
/home/coco/Desktop/c_windows/Users/Koko/Desktop/HELP/25_10_2026/IS2108/flatfair/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


... (40 outputs left out)

Outer refit 2026-05 v5.2_no_geo q10 40


Outer refit 2026-05 v5.2_no_geo q90 40


Completed 2026-05 {"count": 300, "servedCoverage": 0.81, "rawCoverage": 0.81, "medianWidthPercent": 44.9358955201578, "meanWidthPercent": 48.059995604869435, "meanWidthDollars": 306778.12, "mapePercent": 10.792805930776966, "falseHoldRate": 0.15666666666666668, "aboveShare": 0.15666666666666668, "typoShare": 0.0} seconds 0.56


{
  "pooled": {
    "count": 1200,
    "servedCoverage": 0.8191666666666667,
    "rawCoverage": 0.8191666666666667,
    "medianWidthPercent": 49.91732185292753,
    "meanWidthPercent": 53.66090471031555,
    "meanWidthDollars": 327487.43083333335,
    "mapePercent": 11.261256754811196,
    "falseHoldRate": 0.17166666666666666,
    "aboveShare": 0.17166666666666666,
    "typoShare": 0.0
  },
  "weakestSubgroups": [
    {
      "dimension": "town",
      "group": "TOA PAYOH",
      "count": 47,
      "servedCoverage": 0.6170212765957447,
      "rawCoverage": 0.6170212765957447,
      "medianWidthPercent": 63.26715158875791,
      "meanWidthPercent": 65.63108780345624,
      "meanWidthDollars": 429890.44680851063,
      "mapePercent": 17.93303776717815,
      "falseHoldRate": 0.3617021276595745,
      "aboveShare": 0.3617021276595745,
      "typoShare": 0.0
    },
    {
      "dimension": "town",
      "group": "QUEENSTOWN",
      "count": 47,
      "servedCoverage": 0.6595744680851063,
 

In [18]:
print(run_label)

model_file = MODEL_FOLDER / "price_model_v5.joblib"

metrics_hash = show_file("Metrics file:", METRICS_FILE)

model_hash = show_file("Model file:", model_file)

show_file("Price range report:", PRICE_BAND_REPORT)

show_file("Past prices report:", BACKTEST_REPORT)

if metrics_hash:
    show_scores(json.loads(METRICS_FILE.read_text()))

if metrics_hash and model_hash:
    show_model(model_file, METRICS_FILE)

if short_run:
    saved_metrics_now = show_file("Saved metrics file:", SAVED_METRICS)

    serving_model_now = show_file("Serving model:", SERVING_MODEL)

    print("Saved metrics file unchanged:", saved_metrics_now == saved_metrics_hash)
    print("Serving model unchanged:", serving_model_now == serving_model_hash)

Short training check
Metrics file: /home/coco/Desktop/c_windows/Users/Koko/Desktop/HELP/25_10_2026/IS2108/flatfair/.flatfair-local/short-names-20261114T144349Z/notebook-after/app/training/check_runs/price/price_model_v5_metrics.json
  size in bytes: 22981
  SHA256: 92fbf964ec3adb36daed5bbce7909e6f89987a90c534b5ca02a000741ee031da
  written after setup ran: True
Model file: /home/coco/Desktop/c_windows/Users/Koko/Desktop/HELP/25_10_2026/IS2108/flatfair/.flatfair-local/short-names-20261114T144349Z/notebook-after/app/training/check_runs/price/price_model_v5.joblib
  size in bytes: 502542
  SHA256: 405b3d42c5adc6637c3e0e04e6861bce29c1343bb80c820d88d21024738019f7
  written after setup ran: True
Price range report: /home/coco/Desktop/c_windows/Users/Koko/Desktop/HELP/25_10_2026/IS2108/flatfair/.flatfair-local/short-names-20261114T144349Z/notebook-after/app/training/check_runs/price/price_band_eval.json
  size in bytes: 16596
  SHA256: f965d673327aa003c2b11a9af27be10469493af231c7815b80f4da1b30

Serving model: /home/coco/Desktop/c_windows/Users/Koko/Desktop/HELP/25_10_2026/IS2108/flatfair/.flatfair-local/short-names-20261114T144349Z/notebook-after/app/flatfairproj/ml/price_model_v5.joblib
  size in bytes: 29502418
  SHA256: 49db3c767892c5c0418bdd28eb74d67e3dab879262b11dcbe960988e03aa6cf7
  written after setup ran: False
Saved metrics file unchanged: True
Serving model unchanged: True
